## Running this notebook

Install the project with the notebook extra before opening this file. Imports
use the installed package and do not change the working directory. Set
`OVERSCREENING_OUTPUT_DIR` to choose a directory for saved files.

The original scientific parameter values and sweep sizes are retained. Several
cells are deliberately expensive or demonstrate a failed solve; they are not
part of the bounded CI examples. Inspect convergence flags before interpreting
results. Adjoint initialization and unsupported legacy boundary/operator
experiments are explicitly labelled and use the canonical collocation path.
For bounded NumPy/mpmath demonstrations, run the scripts in `examples/`.

Ordinary BSK examples construct a fresh immutable `BSKProblem`; there are no
parameter setters in that interface. The initial problem defaults retain
sigma=10, mu=0, gamma=0.5, delta=50 and wall value 100. Every later parameter
block specifies its physical values and boundary data explicitly. Finite
segments use `domain` and both endpoint boundary rows. Only labelled legacy
operator/boundary branches use the old canonical problem classes.


## This notebook provides a set of examples on how to use the numerical method for the overscreening breakdown problem.

#### The equations under consideration in a general form are:
$$
u_{xx} - \delta^2 u_{xxxx} = f(u, x), x\in[0, \infty)
$$

#### where the right hand side function for the overscreening breakdown problem is formulated as:

$$
f(u,x) :=  \frac{\sinh(u)}{1 +2\gamma \sinh^2\left(u/2\right)} - \frac{0.5 \Phi(x) e^{u}}{1 +2\gamma \sinh^2\left(u/2\right)},
$$


$$
\Phi(x) = \mu \frac{1}{\sqrt{2 \pi}} \exp{\left(-\frac{x^2}{2(\sigma/L_D)^2}\right)}
$$

In [ ]:
# Install the project first: python -m pip install -e ".[notebook]"
from pathlib import Path
import os
from matplotlib import pyplot as plt
import numpy as np
import mpmath as mp
from overscreening_breakdown import BSKProblem, solve, load_solution
from overscreening_breakdown.problems.bsk import overscreening_breakdown as LegacyBSK
from overscreening_breakdown.arithmetic import MpmathArithmetic, NumpyArithmetic
from overscreening_breakdown.rational_mapping import RationalHalfLineMap
# The canonical legacy path is used only for the explicitly labelled adjoint or
# unsupported-operator/boundary experiments retained below.
from overscreening_breakdown.collocation.spectral1d import solve_nonlinear_problem
from overscreening_breakdown.collocation.adapter import CollocationProblemAdapter
from overscreening_breakdown.collocation.solution import CollocationSolution
OUTPUT_DIR = Path(os.environ.get("OVERSCREENING_OUTPUT_DIR", "notebook-output")).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
%matplotlib inline

def status(result):
    print("nonlinear_converged=", result.nonlinear_converged,
          "quadrature_converged=", result.quadrature_converged)

def display_values(function, coordinates):
    # Float conversion is confined to plot data.
    return np.array([float(function(value)) for value in coordinates])

def mapped_values(result, angles):
    a = result.arithmetic
    if result.domain[1] == a.inf:
        mapping = RationalHalfLineMap(a, L=result.L)
        coordinates = [mapping.from_angle(a.scalar(value)) for value in angles]
    else:
        left, right = result.domain
        coordinates = [(a.cos(a.scalar(value))*(right-left)+(right+left))/2 for value in angles]
    return display_values(result.u, coordinates)

def run_collocation(problem, N, *, L=1, tolerance="1e-10", method="newton",
                    max_iterations=50, globalization=True, initial=None,
                    use_mpmath=False, precision=100, adjoint=False):
    if not adjoint:
        if initial is not None and hasattr(initial, "coefficients"):
            initial = (initial.coefficients if initial.N == N
                       else lambda x, a, previous=initial: previous.u(x))
        return solve(problem, N, solver="collocation", L=L, tolerance=tolerance,
                     method=method, max_iterations=max_iterations,
                     use_globalization=globalization, initial_guess=initial,
                     use_mpmath=use_mpmath, precision=precision)
    # Adjoint initialization has no option in the common entry point. Preserve
    # that original experiment through the canonical legacy collocation API,
    # then capture its coefficients in an owned-arithmetic result for evaluation.
    a = MpmathArithmetic(precision) if use_mpmath else NumpyArithmetic()
    physical = problem.bind(a)
    previous_bits = mp.mp.prec
    try:
        legacy = solve_nonlinear_problem(
            N=N, problem=CollocationProblemAdapter(physical), L=L,
            tolerance=tolerance, use_method=method, total_iterations=max_iterations,
            use_globalization=globalization, use_adjoint_opimization=True,
            use_mpmath=use_mpmath, prec=precision)
        coefficients = legacy.solve_problem(None if initial is None else initial.coefficients)
        return CollocationSolution(physical=physical, coefficients=coefficients, N=N, L=L,
                                   method=method, converged=legacy.is_converged(),
                                   newton_result=legacy.get_newton_result())
    finally:
        mp.mp.prec = previous_bits


In [ ]:
#init the problem

problem = BSKProblem()

In [ ]:
# Numerical parameters are unchanged from the original notebook.
N = 100
domain = [0,np.inf]
tolerance = 1.0e-10
use_method = "newton"
use_adjoint_opimization = False
L = 6


### On how to select the parameter $L$, see [bellow](#check_L_values).
But we recommend exploring all cells to be certain on what you are doing.

In [ ]:
#set up the problem.
#If a parameter is not given, then a default value is used instead. 
#See the source file of "overscreening_breakdown" for more details for defauls values

sigma = 20.0
mu = 0.36
gamma = 1
delta = 10.0
u0 = 1.0
u0xxx = 0

#use some function as initial guess for the root in the Newton's method.
def some_func(x, a):
    return u0 * a.exp(-x)

#apply parameters
problem = BSKProblem(sigma=sigma, mu=mu, gamma=gamma, delta=delta, boundary_conditions=((u0, None, None, u0xxx, None), (None,)*5), initial_guess=some_func)

In [ ]:
# The physical description is passed directly to solve; no solver mutation is needed.
print(problem.domain, problem.boundary_conditions)


In [ ]:
result = run_collocation(problem, N, L=L, tolerance=tolerance, method=use_method)
c = result.coefficients
status(result)


In [ ]:
# Explicit canonical legacy path: retain the original adjoint initialization comparison.
use_adjoint_opimization = True
result_ao = run_collocation(problem, N, L=L, tolerance=tolerance,
                            method=use_method, adjoint=use_adjoint_opimization)
c_ao = result_ao.coefficients
status(result_ao)


In [ ]:
#check solution relative difference
np.linalg.norm(c_ao - c)/np.linalg.norm(c)

In [ ]:
# Typed JSON records solver type, basis, mapping, precision and physical metadata.
path = OUTPUT_DIR / "solution.json"
result.save(path)
restored = load_solution(path, problem=problem, solver="collocation")
print("saved coefficient difference:", result.arithmetic.norm(result.coefficients-restored.coefficients))
# Same-solver restart; use project_collocation explicitly before a Galerkin restart.
restarted = solve(problem, N, solver="collocation", L=L, tolerance=tolerance,
                  initial_guess=restored)
status(restarted)


In [ ]:
t = np.arange(0, np.pi, 0.01)
plt.title(r"solution on the mapped segment $[0,\pi]$")
plt.plot(t, mapped_values(result, t), '.')
plt.plot(t, mapped_values(result_ao, t))
plt.legend(["no adjoint initialization", "adjoint initialization"])
plt.show()
x = np.arange(0.0, 100.0, 0.1)
plt.title(r"solution on part of $\mathbb{R}^+$")
plt.plot(x, display_values(result.u, x), '.')
plt.plot(x, display_values(result_ao.u, x))
plt.legend(["no adjoint initialization", "adjoint initialization"])
plt.show()


Next we demonstrate the applicaiton of the size globalization for a very stiff problem using globalization in the size of the problem thanks to the natural behaviour of spectral methods.

In [ ]:
#we are setting initial parameters to the problem for this test, specially desiged to be very stiff
sigma = 10.0
mu = 1
gamma = 1/2
delta = 50.0
u0 = 100.0
u0xxx = 0

#initial guess should be u0*np.exp(-x**2). This breaks the problem convergence
def some_func(x, a):
    return a.exp(-x ** 2) 

problem = BSKProblem(sigma=sigma, mu=mu, gamma=gamma, delta=delta, boundary_conditions=((u0, None, None, u0xxx, None), (None,)*5), initial_guess=some_func)

In [ ]:
# This deliberately stiff solve may fail; inspect the status rather than assuming success.
N = 100
result = run_collocation(problem, N, L=L, tolerance=tolerance, method=use_method,
                         max_iterations=20, globalization=False,
                         adjoint=use_adjoint_opimization)
c = result.coefficients
status(result)


In [ ]:
# Original 120-digit, N=100 experiment; it can be expensive.
print("using u0 =", u0)
def some_func_mp(x, a):
    return a.exp(-x ** 2)
problem_mp = BSKProblem()
problem_mp = BSKProblem(sigma=sigma, mu=mu, gamma=gamma, delta=delta, boundary_conditions=((u0, None, None, u0xxx, None), (None,)*5), initial_guess=some_func_mp)
N = 100
use_mpmath = True
prec = 120
tolerance_mp = 1.0e-30
result_mp = run_collocation(problem_mp, N, L=L, tolerance=tolerance_mp, method=use_method,
                            max_iterations=20, globalization=False, use_mpmath=use_mpmath,
                            precision=prec, adjoint=use_adjoint_opimization)
c_mp = result_mp.coefficients
status(result_mp)


In [ ]:
t = np.arange(0, np.pi, 0.01)
x = np.arange(0, 100, 0.1)
plt.title(r"solution on the mapped segment $[0,\pi]$")
plt.plot(t, mapped_values(result_mp, t))
plt.show()
plt.title(r"solution on part of $\mathbb{R}^+$")
plt.plot(x, display_values(result_mp.u, x))
plt.show()


In [ ]:
# Original size-globalization sweep; changing N transfers the previous field.
N_base = 10
N = N_base
result1 = None
previous_result = None
it = 2
while it < 12:
    print("running on N =", N)
    result = run_collocation(problem, N, L=L, tolerance=tolerance, method=use_method,
                             max_iterations=20, globalization=False, initial=result1)
    status(result)
    if result.converged:
        previous_result = result1
        result1 = result
        N = N_base*it
        it = it+1
    else:
        N = N-5
c1 = result1.coefficients


In [ ]:
t = np.arange(0, np.pi, 0.01)
x = np.arange(0, 60, 0.1)
comparisons = [("high precision solution", result_mp),
               ("previous accepted size", previous_result), ("last accepted size", result1)]
for label, current in comparisons:
    if current is not None:
        plt.plot(t, mapped_values(current, t), label="{}, N={}".format(label, current.N))
plt.title(r"comparison on $[0,\pi]$")
plt.legend()
plt.show()
for label, current in comparisons:
    if current is not None:
        plt.plot(x, display_values(current.u, x), label="{}, N={}".format(label, current.N))
plt.title(r"comparison on part of $\mathbb{R}^+$")
plt.legend()
plt.show()


Next we demonstrate the applicaiton of the homotopy* globalization for the same stiff problem.

*https://en.wikipedia.org/wiki/Homotopy

In [ ]:
# Same N=100 and 20-iteration budget; only globalization is enabled.
globalization_options = dict(N=100, L=L, tolerance=tolerance, method="newton",
                              max_iterations=20, globalization=True,
                              adjoint=use_adjoint_opimization)


In [ ]:
result2 = run_collocation(problem, **globalization_options)
c2 = result2.coefficients
status(result2)


In [ ]:
t = np.arange(0, np.pi, 0.01)
sol = mapped_values(result1, t)
sol2 = mapped_values(result2, t)
plt.title(r"size versus homotopy globalization on $[0,\pi]$")
plt.plot(t, sol2, '.')
plt.plot(t, sol)
plt.show()
plt.title(r"difference of the two computed fields on $[0,\pi]$")
plt.plot(t, sol-sol2)
plt.show()


<a id='check_L_values'></a>
## Here we demonstrate how to find a stable value of the mapping parameter L for the given problem.

One should choose the smallest possible value of L at which the solutions are stable w.r.t. the discretization


In [ ]:
#setup desired problem parameters for the test

sigma = 10.0
mu = 1
gamma = 1/2
delta = 10.0
u0 = 1.0
u0xxx = 0

#solution initial guess
def some_func(x, a):
    return u0 * a.exp(-x) 

problem = BSKProblem(sigma=sigma, mu=mu, gamma=gamma, delta=delta, boundary_conditions=((u0, None, None, u0xxx, None), (None,)*5), initial_guess=some_func)

In [ ]:
# Original resolution and mapping-length grids are preserved.
Nall = [30,70,80,100]
Lall = [1,2,3,4,5,6]
t = np.arange(0, np.pi, 0.01)
x = np.arange(0, 100, 0.1)
use_adjoint_opimization = True
for L in Lall:
    results = {}
    previous = None
    for N in Nall:
        current = run_collocation(problem, N, L=L, tolerance=tolerance, method="newton",
                                  max_iterations=20, globalization=False, initial=previous)
        status(current)
        results[N] = current
        previous = current
    for N in Nall:
        plt.plot(t, mapped_values(results[N], t), label=N)
    plt.legend()
    plt.title("mapped segment, L="+str(L))
    plt.show()
    for N in Nall:
        plt.plot(x, display_values(results[N].u, x), label=N)
    plt.legend()
    plt.title(r"$\mathbb{R}^+$, L="+str(L))
    plt.show()
print("For this problem the original recommendation was L=5 or L=6.")


In the section bellow we demonstrate how to solve the Poisson problem in the form:
$$
u_{xx}=f(x,u), x\in[0, \infty)
$$

In [ ]:
sigma = 1.0
mu = 1
gamma = 1
delta = 0.0
u0 = 1.0
u0xxx = None  # The common second-order formulation needs no third-derivative wall condition.
def some_func(x, a):
    return u0 * a.exp(-x)
problem = BSKProblem()
problem = BSKProblem(sigma=sigma, mu=mu, gamma=gamma, delta=delta, boundary_conditions=((u0, None, None, u0xxx, None), (None,)*5), initial_guess=some_func)
poisson = run_collocation(problem, N, L=L, tolerance=tolerance, method=use_method)
status(poisson)
t = np.arange(0, np.pi, 0.01)
plt.title(r"second-order solution on $[0,\pi]$")
plt.plot(t, mapped_values(poisson, t))
plt.show()
x = np.arange(0.0, 10.0, 0.1)
plt.title(r"second-order solution on part of $\mathbb{R}^+$")
plt.plot(x, display_values(poisson.u, x))
plt.show()


### Bellow some additional tests are provided.

Feel free to test your parameters as well.

In [ ]:
#set up problem.
#If a parameter is not given, then a default value is used instead. See the source file of "overscreening_breakdown" for more details

sigma = 20.0
mu = 0.0
gamma = 0.1
delta = 0.0
u0 = 0.1
u0xxx = 0

def some_func(x):
    return u0*np.exp(-x**4)

problem = LegacyBSK()  # Explicit historical unsupported wall-data experiment.
problem.set_parameters(sigma = sigma, mu = mu, gamma = gamma, delta = delta, u0 = u0, u0xxx=u0xxx, initial_guess_solution=some_func)

In [ ]:
# Historical unsupported boundary experiment: delta=0 with a third-derivative wall condition.
# Keep the explicit canonical legacy collocation API for this section.
solver = solve_nonlinear_problem(N=N, domain=[0,np.inf], tolerance=tolerance,
                                 use_method=use_method, use_adjoint_opimization=use_adjoint_opimization)
solver.set_mapping_parameter(L)
solver.set_problem(problem)
c = solver.solve_problem()
#see the solution
t = np.arange(0, np.pi, 0.01)
sol = solver.obtain_solution_in_basis(c, t)
plt.title(r"solution on the mapped segment [0,$\pi$]")
plt.plot(t, sol)
plt.show()
x = np.arange(0.0, 40.0, 0.1)
sol = solver.obtain_solution_in_domain(c, x)
plt.title(r"solution on the part of the $\mathbb{R}^+$")
plt.plot(x, sol)
plt.show()

In [ ]:
# we perfrom a set of iterations to update a better initial guess from the smaller problem size
def some_func(x):
    return u0*np.exp(-x**4)

L = 1
tolerance = 1.0e-10
N = 60
sigma = 20.0
mu = 0.0
gamma = 1.0
delta = 0.0
u0 = 10.0
u0xxx = 0
u0_c = 0.1
c1 = None
alpha = 1.0

def some_func(x):
    return 2*np.log((1.0+np.tanh(u0/4.0)*np.exp(-x))/(1.0-np.tanh(u0/4.0)*np.exp(-x)))

for it in range(1,1000):
    print("running on gamma = ", gamma)
    solver = solve_nonlinear_problem(N=N, domain=[0,np.inf], tolerance=tolerance, use_method=use_method, total_iterations = 100, use_globalization = False)
    solver.set_mapping_parameter(L)
    problem = LegacyBSK()  # Explicit historical unsupported wall-data experiment.
    problem.set_parameters(sigma = sigma, mu = mu, gamma = gamma, delta = delta, u0 = u0, u0xxx=u0xxx, initial_guess_solution=some_func)
    solver.set_problem(problem)
    c = solver.solve_problem(c1)
    if solver.is_converged():
        c1 = np.copy(c)
        gamma_old = gamma
        gamma = gamma*0.5
        gamma_found = gamma;
        alpha = 1.0
    else:
        alpha = alpha*0.5
        gamma = (1-alpha)*gamma_old + alpha*gamma_found
    if gamma == 0.0:
        break
    if gamma<1.0e-12:
        gamma = 0.0

In [ ]:
# we apply homotopy when solution fails to converge for some bad RHS
def some_func(x):
    return u0*np.exp(-x**2)
L = 1
tolerance = 1.0e-10
N = 100
sigma = 20.0
mu = 0.0
gamma = 0.0

delta = 0.0
u0 = 2.0
u0xxx = 0
def some_func(x):
    return 2*np.log((1.0+np.tanh(u0/4.0)*np.exp(-x))/(1.0-np.tanh(u0/4.0)*np.exp(-x)))

solver = solve_nonlinear_problem(N=N, domain=[0,np.inf], tolerance=tolerance, use_method=use_method, total_iterations = 10, use_globalization = True)
solver.set_mapping_parameter(L)
problem = LegacyBSK()  # Explicit historical unsupported wall-data experiment.
problem.set_parameters(sigma = sigma, mu = mu, gamma = gamma, delta = delta, u0 = u0, u0xxx=u0xxx, initial_guess_solution=some_func)
solver.set_problem(problem)
c = solver.solve_problem()

In [ ]:
#see the solution
t = np.arange(0, np.pi, 0.01)
sol = solver.obtain_solution_in_basis(c, t)
plt.title(r"solution on the mapped segment [0,$\pi$]")
plt.plot(t, sol)
plt.show()
x = np.arange(0.0, 5.0, 0.01)
sol = solver.obtain_solution_in_domain(c, x)
plt.title(r"solution on the part of the $\mathbb{R}^+$")
plt.plot(x, sol)
plt.show()